# Discovery dataset - MRS quality control


In [1]:
from pathlib import Path
import os
import pandas as pd
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

In [4]:
df = pd.read_csv(Path(base_path) / "DATA/discovery_dataset/20260529_101516_WAND_ses-05_GANNET_MRS_results.csv")

In [ ]:
THRESH_SNR = 5 # SNR must be ABOVE this
THRESH_FWHM = 30 # FWHM must be BELOW this
THRESH_FITERROR = 15 #FitError must be BELOW

df['GABA_eligible'] = (df['GABA_FitError'] < THRESH_FITERROR) & (df['GABA_FWHM'] < THRESH_FWHM) & (df['GABA_SNR'] > THRESH_SNR)
df['Glx_eligible'] = (df['Glx_FitError'] < THRESH_FITERROR) & (df['Glx_FWHM'] < THRESH_FWHM) & (df['Glx_SNR'] > THRESH_SNR)
df['Overall_eligible'] = df['GABA_eligible'] & df['Glx_eligible']

In [6]:
print(df['Overall_eligible'].value_counts())

Overall_eligible
True     608
False      4
Name: count, dtype: int64


In [7]:
display(df[df['Overall_eligible'] == False])

,Subject,Voxel,GABA_ConcIU_TissCorr,GABA_ConcIU_AlphaTissCorr,Glx_ConcIU_TissCorr,Glx_ConcIU_AlphaTissCorr,GlxGABA_TissCorr,GlxGABA_AlphaTissCorr,GABA_FWHM,GABA_SNR,GABA_FitError,Glx_FWHM,Glx_SNR,Glx_FitError,fGM,WaterSuppFactor,Status,GABA_eligible,Glx_eligible,Overall_eligible
205,sub-38989,anteriorcingulate,2.614541,2.327575,8.11108972740697+0i,7.2208340299889+0i,3.10229948954423+0i,3.10229948954423+0i,26.542005,12.786759,16.925798,18.459510,17.720058,4.694052,0.600005,NaN,ok,False,True,False
307,sub-54552,rightauditory,NaN,NaN,NaN+0i,NaN+0i,NaN+0i,NaN+0i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skipped: no ref,False,False,False
412,sub-72783,occipital,NaN,NaN,NaN+0i,NaN+0i,NaN+0i,NaN+0i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skipped: content not downloaded,False,False,False
425,sub-73512,rightauditory,-5.750268,-5.208452,-20.6056018861388+0.0523866780396484i,-18.6640489871975+0.0474505685692326i,3.58341570653965-0.00911030145773084i,3.58341570653965-0.00911030145773083i,26.183591,20.015926,-2.177544,67.454878,0.026632,-2570.080668,0.584714,NaN,ok,True,False,False


In [8]:
cleaned_df = df[df['Overall_eligible'] == True].copy()

exclusions_path = (Path(base_path) / "DATA/discovery_dataset/exclusions.csv") # voxels to exclude based on visual inspection

exclusions = pd.read_csv(exclusions_path)
exclusions.columns = exclusions.columns.str.strip().str.lower()
exclusions = exclusions.rename(columns={"subject": "Subject", "voi": "Voxel"})[["Subject", "Voxel"]].drop_duplicates()

cleaned_df = df[df["Overall_eligible"] == True].copy()
cleaned_df = cleaned_df.merge(exclusions.assign(_exclude=True), on=["Subject", "Voxel"], how="left")
cleaned_df = cleaned_df.loc[cleaned_df["_exclude"].ne(True)].drop(columns="_exclude").copy()

print(f"cleaned_df rows after exclusions: {len(cleaned_df)}")

cleaned_df rows after exclusions: 605


In [9]:
# Convert complex strings to float

def complex_i_to_float(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    z = complex(x.replace("i", "j"))
    return z.real

cols_to_convert = ["Glx_ConcIU_AlphaTissCorr", "GlxGABA_AlphaTissCorr"]

for col in cols_to_convert:
    cleaned_df[col] = cleaned_df[col].apply(complex_i_to_float)

print(cleaned_df.dtypes)

Subject                          str
Voxel                            str
GABA_ConcIU_TissCorr         float64
GABA_ConcIU_AlphaTissCorr    float64
Glx_ConcIU_TissCorr              str
Glx_ConcIU_AlphaTissCorr     float64
GlxGABA_TissCorr                 str
GlxGABA_AlphaTissCorr        float64
GABA_FWHM                    float64
GABA_SNR                     float64
GABA_FitError                float64
Glx_FWHM                     float64
Glx_SNR                      float64
Glx_FitError                 float64
fGM                          float64
WaterSuppFactor              float64
Status                           str
GABA_eligible                   bool
Glx_eligible                    bool
Overall_eligible                bool
dtype: object


In [10]:
out_dir = str(Path(base_path) / "DATA/discovery_dataset")
out_csv = os.path.join(out_dir, "cleaned_MRS_df.csv")

cleaned_df.to_csv(out_csv, index=False)

print(f"Saved cleaned_df to: {out_csv}")

Saved cleaned_df to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/DATA/discovery_dataset/cleaned_MRS_df.csv
